# 04.1 — Painel de Avaliação dos Modelos CVM

Visualização interativa dos resultados gerados pelo **Script 4** (`04_cvm_avaliacao.ipynb`).

**O que este script entrega:**
- Resumo executivo do desempenho por variável e horizonte
- Evolução temporal do Z''-Score por empresa e setor
- Distribuição de incerteza Monte Carlo por empresa
- Intervalos Conformais: cobertura real vs. esperada
- Feature importance: quais variáveis explicam as previsões
- Score de risco composto: visão consolidada por setor
- Painel comparativo de predito × observado para os targets foco do TCC

**Pré-requisito:** executar `04_cvm_avaliacao.ipynb` completo antes deste script.

In [2]:
import warnings
import logging
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

PASTA_SAIDA  = Path('outputs')
PASTA_PAINEL = PASTA_SAIDA / 'painel_avaliacao'
PASTA_PAINEL.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO,
)
logger = logging.getLogger('painel_avaliacao')

# Paleta consistente com o Script 4
PALETA_ALG = {
    'Ridge': '#3498db', 'SVR': '#9b59b6',
    'RandomForest': '#2ecc71', 'GradientBoosting': '#e74c3c',
    'Ensemble': '#f39c12',
}
PALETA_SETOR = {
    'Commodities': '#e74c3c', 'Energia': '#3498db',
    'Petroleo': '#f39c12',    'Tecnologia': '#2ecc71',
    'Varejo': '#9b59b6',
}
ZONA_SEGURA    = 2.60
ZONA_CINZA_INF = 1.10
NOME_VARIAVEL  = {
    'DRE_3.01': 'Receita Liquida',  'DRE_3.11': 'Lucro Liquido',
    'EBITDA':   'EBITDA',           'BPA_1':    'Ativo Total',
    'BPA_1.01': 'Ativo Circulante', 'BPP_2.01': 'Passivo Circulante',
    'BPP_2.03': 'Patrimonio Liquido','BPP_2':   'Passivo Total',
    'DFC_MI_6.01': 'FCO',
}

def _salvar(fig, nome, pasta=PASTA_PAINEL):
    caminho = pasta / nome
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.close(fig)
    logger.info('Figura salva: %s', caminho)
    return caminho

print('OK Configuracao concluida')
print(f'Figuras serao salvas em: {PASTA_PAINEL}')


OK Configuracao concluida
Figuras serao salvas em: outputs\painel_avaliacao


##  Bloco 1 — Carregamento dos Artefatos do Script 4

In [3]:
def _load(nome, req=False):
    p = PASTA_SAIDA / nome
    if not p.exists():
        if req:
            raise FileNotFoundError(f'{nome} nao encontrado. Execute o Script 4 antes.')
        logger.warning('%s nao encontrado — etapa correspondente sera ignorada.', nome)
        return None
    try:
        if nome.endswith('.parquet'):
            df = pd.read_parquet(p)
        elif nome.endswith('.csv'):
            df = pd.read_csv(p)
        elif nome.endswith('.pkl'):
            import pickle
            with open(p, 'rb') as f:
                df = pickle.load(f)
        else:
            raise ValueError(f'Extensao nao suportada: {nome}')
        logger.info('Carregado: %s | %s', nome,
                    f'{len(df):,} linhas' if hasattr(df, '__len__') else type(df).__name__)
        return df
    except Exception as e:
        logger.error('Erro ao carregar %s: %s', nome, e)
        return None

# ── Carregamento ──────────────────────────────────────────────────────────────
df_te          = _load('resultados_teste_enriquecido.csv')
df_cv          = _load('resultados_cv.csv')
df_diag        = _load('diagnostico_overfitting.csv')
df_setor       = _load('metricas_por_setor.csv')
df_zscore      = _load('altman_zscore.parquet')
df_stress      = _load('analise_estresse_mc.csv')
df_feat        = _load('feature_importance_ranking.csv')
df_cp          = _load('conformal_summary.csv')
df_pred        = _load('predicoes_teste_detalhadas.parquet')
df_res         = _load('residuos_resumo.csv')
melhores       = _load('melhores_modelos.pkl') or {}
zscore_emp     = _load('zscore_por_empresa.pkl') or {}
emps_destaque  = _load('empresas_destaque.pkl') or {}

# Mapas de lookup
dataset = None
for _nome in ('dataset_cvm_consolidado.parquet',):
    _p = PASTA_SAIDA / _nome
    if _p.exists():
        dataset = pd.read_parquet(_p)
        break

mapa_nome = {}; mapa_setor = {}
if dataset is not None and 'CNPJ_CIA' in dataset.columns:
    _b = dataset.drop_duplicates('CNPJ_CIA')
    if 'NOME_CIA' in _b.columns:
        mapa_nome  = _b.set_index('CNPJ_CIA')['NOME_CIA'].to_dict()
    if 'SETOR' in _b.columns:
        mapa_setor = _b.set_index('CNPJ_CIA')['SETOR'].to_dict()

print()
print('=== Resumo dos artefatos carregados ===')
artefatos = {
    'resultados_teste_enriquecido': df_te,
    'diagnostico_overfitting':      df_diag,
    'metricas_por_setor':           df_setor,
    'altman_zscore':                df_zscore,
    'analise_estresse_mc':          df_stress,
    'feature_importance_ranking':   df_feat,
    'conformal_summary':            df_cp,
    'predicoes_teste_detalhadas':   df_pred,
    'residuos_resumo':              df_res,
}
for nome, obj in artefatos.items():
    if obj is not None and hasattr(obj, '__len__'):
        status = f'{len(obj):>6,} linhas'
    elif obj is None:
        status = '  AUSENTE'
    else:
        status = '  OK'
    print(f'  {nome:<38} {status}')
print(f'  melhores_modelos                       {len(melhores):>6} targets')
print(f'  empresas_destaque                      {len(emps_destaque):>6} setores')
print(f'  zscore_por_empresa                     {len(zscore_emp):>6} empresas')


2026-06-07 23:21:04 | INFO     | Carregado: resultados_teste_enriquecido.csv | 180 linhas
2026-06-07 23:21:04 | INFO     | Carregado: resultados_cv.csv | 180 linhas
2026-06-07 23:21:04 | INFO     | Carregado: diagnostico_overfitting.csv | 36 linhas
2026-06-07 23:21:04 | INFO     | Carregado: metricas_por_setor.csv | 180 linhas
2026-06-07 23:21:04 | INFO     | Carregado: altman_zscore.parquet | 1,031 linhas
2026-06-07 23:21:04 | INFO     | Carregado: analise_estresse_mc.csv | 225 linhas
2026-06-07 23:21:04 | INFO     | Carregado: feature_importance_ranking.csv | 261 linhas
2026-06-07 23:21:04 | INFO     | Carregado: conformal_summary.csv | 36 linhas
2026-06-07 23:21:04 | INFO     | Carregado: predicoes_teste_detalhadas.parquet | 28,035 linhas
2026-06-07 23:21:04 | INFO     | Carregado: residuos_resumo.csv | 36 linhas
2026-06-07 23:21:04 | INFO     | Carregado: melhores_modelos.pkl | 36 linhas
2026-06-07 23:21:04 | INFO     | Carregado: zscore_por_empresa.pkl | 25 linhas
2026-06-07 23:21


=== Resumo dos artefatos carregados ===
  resultados_teste_enriquecido              180 linhas
  diagnostico_overfitting                    36 linhas
  metricas_por_setor                        180 linhas
  altman_zscore                           1,031 linhas
  analise_estresse_mc                       225 linhas
  feature_importance_ranking                261 linhas
  conformal_summary                          36 linhas
  predicoes_teste_detalhadas             28,035 linhas
  residuos_resumo                            36 linhas
  melhores_modelos                           36 targets
  empresas_destaque                           5 setores
  zscore_por_empresa                         25 empresas


##  Bloco 2 — Resumo Executivo: SMAPE por Variável × Horizonte

Visão consolidada do desempenho no hold-out. A tabela mostra o SMAPE médio macro-empresa
do melhor algoritmo para cada combinação variável × horizonte.

**Como ler:** valores mais baixos = melhor previsão. SMAPE = 0,15 significa erro médio de 15%.

In [4]:
if df_te is not None and 'nome_variavel' in df_te.columns:
    # Filtra melhor modelo por target
    if 'melhor_target' in df_te.columns:
        df_m = df_te[df_te['melhor_target'] == True].copy()
    else:
        df_m = df_te.copy()

    _col_smape = next((c for c in df_te.columns if 'SMAPE_teste' in c), None)
    _col_hor   = 'Horizonte' if 'Horizonte' in df_te.columns else None

    if _col_smape and _col_hor:
        # Limpa o nome do horizonte
        df_m['Horizonte_clean'] = df_m[_col_hor].str.replace('_', '')

        pv = df_m.pivot_table(
            values=_col_smape,
            index='nome_variavel',
            columns='Horizonte_clean',
            aggfunc='mean'
        )
        # Ordena colunas por horizonte cronologico
        col_order = [c for c in ['ITRT1','ITRT2','ITRT3','DFP'] if c in pv.columns]
        pv = pv[col_order]

        print('=== SMAPE medio (melhor modelo) por Variavel x Horizonte ===')
        print('(valores menores = melhor desempenho | 0.10 = 10% de erro medio)')
        print()
        print(pv.round(3).to_string())

        # Figura 1 — Heatmap
        fig, ax = plt.subplots(figsize=(9, 5))
        vmax = min(float(pv.max().max()), 1.0)
        sns.heatmap(
            pv, annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, linecolor='white', ax=ax,
            vmin=0, vmax=vmax,
            cbar_kws={'label': 'SMAPE macro-empresa'}
        )
        ax.set_title(
            'SMAPE por Variavel x Horizonte\n(melhor modelo por target, hold-out 2024-2025)',
            fontsize=11, fontweight='bold'
        )
        ax.set_xlabel('Horizonte', fontsize=9)
        ax.set_ylabel('')
        plt.tight_layout()
        _salvar(fig, 'p41_b2_smape_heatmap.png')
        print()
        print('OK Figura salva: p41_b2_smape_heatmap.png')

        # Ranking de algoritmos vencedores
        if melhores:
            from collections import Counter
            cnt = Counter(melhores.values())
            print()
            print('=== Algoritmos vencedores (por nr de targets ganhos) ===')
            for alg, n in sorted(cnt.items(), key=lambda x: -x[1]):
                pct = n / len(melhores) * 100
                barra = chr(9608) * int(pct / 3)
                print(f'  {alg:<22} {n:>3}x ({pct:5.1f}%)  {barra}')
else:
    print('AVISO: resultados_teste_enriquecido.csv nao disponivel.')


=== SMAPE medio (melhor modelo) por Variavel x Horizonte ===
(valores menores = melhor desempenho | 0.10 = 10% de erro medio)

Horizonte_clean     ITRT1  ITRT2  ITRT3    DFP
nome_variavel                                 
Ativo Circulante   0.0970 0.0860 0.0760 0.1810
Ativo Total        0.0770 0.0610 0.0510 0.1670
EBITDA             0.2820 0.1550 0.0870 0.1020
FCO                1.0000 1.0000 1.0000 0.6550
Lucro Líquido      0.6080 0.7750 0.7640 0.4690
Passivo Circulante 0.0960 0.1310 0.1020 0.2090
Passivo Total      0.0770 0.0610 0.0510 0.1670
Patrimônio Líquido 0.0800 0.0560 0.0610 0.1730
Receita Líquida    0.1110 0.1000 0.1190 0.1940


2026-06-07 23:21:04 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b2_smape_heatmap.png



OK Figura salva: p41_b2_smape_heatmap.png

=== Algoritmos vencedores (por nr de targets ganhos) ===
  GradientBoosting        29x ( 80.6%)  ██████████████████████████
  RandomForest             7x ( 19.4%)  ██████


##  Bloco 3 — Diagnóstico de Overfitting (CV vs. Teste)

Compara o desempenho no cross-validation (dados de treino) com o desempenho no hold-out
(dados reais não vistos). **Δ > 0.10** indica possível overfitting.

In [5]:
if df_diag is not None and not df_diag.empty:
    # Limpa nomes de horizonte
    if 'Horizonte' in df_diag.columns:
        df_diag['Horizonte_clean'] = df_diag['Horizonte'].str.replace('_', '')
    else:
        df_diag['Horizonte_clean'] = 'N/A'

    print('=== Diagnostico de Overfitting — Delta = SMAPE_Teste - SMAPE_CV ===')
    print('Delta > 0.10 = possivel overfitting | Delta < 0 = boa generalizacao')
    print()

    # Tabela resumo por variavel
    n_over = (df_diag['Delta'] > 0.10).sum() if 'Delta' in df_diag.columns else 0
    n_ok   = (df_diag['Delta'] <= 0.10).sum() if 'Delta' in df_diag.columns else 0
    print(f'  Sem overfitting (Delta <= 0.10): {n_ok} targets')
    print(f'  Com overfitting (Delta >  0.10): {n_over} targets')
    print()

    if 'SMAPE_CV' in df_diag.columns and 'SMAPE_Teste' in df_diag.columns:
        # Figura 2 — Scatter CV vs Teste
        fig, ax = plt.subplots(figsize=(8, 6))
        sc = ax.scatter(
            df_diag['SMAPE_CV'], df_diag['SMAPE_Teste'],
            c=df_diag['Delta'].clip(-0.2, 0.3),
            cmap='RdYlGn_r', s=60, alpha=0.8, edgecolors='white', lw=0.5
        )
        lim = max(df_diag['SMAPE_CV'].max(), df_diag['SMAPE_Teste'].max()) * 1.05
        ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='CV = Teste')
        ax.plot([0, lim], [0.10, lim + 0.10], 'r:', lw=1, alpha=0.5, label='Delta = +0.10')
        plt.colorbar(sc, ax=ax, label='Delta (Teste - CV)')
        ax.set_xlabel('SMAPE no Cross-Validation', fontsize=10)
        ax.set_ylabel('SMAPE no Hold-Out (Teste)', fontsize=10)
        ax.set_title(
            'Overfitting: SMAPE CV vs. Teste\n'
            '(acima da linha tracejada = degradacao no dado real)',
            fontsize=11, fontweight='bold'
        )
        ax.legend(fontsize=8)
        ax.grid(alpha=0.25)
        plt.tight_layout()
        _salvar(fig, 'p41_b3_overfitting_scatter.png')
        print('OK Figura salva: p41_b3_overfitting_scatter.png')

    # Exibe targets com maior Delta
    if 'Delta' in df_diag.columns and 'Variavel' in df_diag.columns:
        top_over = df_diag.nlargest(10, 'Delta')[['Variavel', 'Horizonte_clean', 'SMAPE_CV', 'SMAPE_Teste', 'Delta', 'TheilU']]
        print()
        print('Top-10 targets com maior Delta (risco de overfitting):')
        print(top_over.round(4).to_string(index=False))
else:
    print('AVISO: diagnostico_overfitting.csv nao disponivel.')


=== Diagnostico de Overfitting — Delta = SMAPE_Teste - SMAPE_CV ===
Delta > 0.10 = possivel overfitting | Delta < 0 = boa generalizacao

  Sem overfitting (Delta <= 0.10): 35 targets
  Com overfitting (Delta >  0.10): 1 targets



2026-06-07 23:21:05 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b3_overfitting_scatter.png


OK Figura salva: p41_b3_overfitting_scatter.png

Top-10 targets com maior Delta (risco de overfitting):
          Variavel Horizonte_clean  SMAPE_CV  SMAPE_Teste  Delta  TheilU
            EBITDA           ITRT1    0.1542       0.2823 0.1281  0.3299
       Ativo Total             DFP    0.1052       0.1669 0.0617  2.0000
     Passivo Total             DFP    0.1052       0.1669 0.0617  2.0000
Passivo Circulante             DFP    0.1511       0.2090 0.0579  2.0000
     Lucro Líquido           ITRT2    0.7169       0.7748 0.0579  0.7696
  Ativo Circulante             DFP    0.1259       0.1805 0.0547  2.0000
   Receita Líquida             DFP    0.1521       0.1938 0.0417  2.0000
Patrimônio Líquido             DFP    0.1345       0.1726 0.0381  2.0000
               FCO           ITRT3    0.9744       1.0000 0.0256  1.0755
               FCO           ITRT1    0.9939       1.0000 0.0061  1.0706


##  Bloco 4 — Desempenho por Setor

Compara a previsibilidade entre setores. Setores com SMAPE mais baixo são mais previsíveis
e têm padrões financeiros mais consistentes ao longo do tempo.

In [6]:
if df_setor is not None and not df_setor.empty:
    print('=== SMAPE medio por Setor x Variavel ===')

    if 'SMAPE' in df_setor.columns and 'SETOR' in df_setor.columns:
        # Pivô setor × variável
        if 'Variavel' in df_setor.columns:
            pv_s = df_setor.pivot_table(values='SMAPE', index='Variavel', columns='SETOR', aggfunc='mean')
            print(pv_s.round(3).to_string())
            print()

        # Ranking de setores
        rank_s = df_setor.groupby('SETOR')['SMAPE'].mean().sort_values()
        print('Ranking de previsibilidade por setor (SMAPE medio):')
        for s, v in rank_s.items():
            barra = chr(9608) * int((1 - min(v, 1)) * 25)
            print(f'  {s:<18} SMAPE={v:.3f}  {barra}')

    if 'TheilU' in df_setor.columns and 'SETOR' in df_setor.columns:
        print()
        print('U de Theil medio por setor (< 1 = supera naive):')
        print(df_setor.groupby('SETOR')['TheilU'].agg(['mean','median']).round(3).to_string())

    # Figura 3 — Boxplot SMAPE por setor
    if 'SMAPE' in df_setor.columns and 'SETOR' in df_setor.columns:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # SMAPE
        sns.boxplot(data=df_setor, x='SETOR', y='SMAPE',
                    order=rank_s.index, palette='RdYlGn_r', ax=axes[0])
        axes[0].set_title('SMAPE por Setor\n(todos os targets, melhor modelo)',
                           fontsize=10, fontweight='bold')
        axes[0].set_xlabel('')
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
        axes[0].axhline(0.20, ls='--', color='orange', lw=1, alpha=0.7, label='SMAPE=20%')
        axes[0].legend(fontsize=8)
        axes[0].grid(axis='y', alpha=0.3)

        # U de Theil
        if 'TheilU' in df_setor.columns:
            df_u = df_setor[df_setor['TheilU'].notna() & (df_setor['TheilU'] < 5)]
            sns.boxplot(data=df_u, x='SETOR', y='TheilU',
                        order=rank_s.index, palette='RdYlGn_r', ax=axes[1])
            axes[1].axhline(1.0, ls='--', color='black', lw=1.5, label='U=1 (naive)')
            axes[1].set_title('U de Theil por Setor\n(< 1 = supera naive)',
                               fontsize=10, fontweight='bold')
            axes[1].set_xlabel('')
            axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
            axes[1].legend(fontsize=8)
            axes[1].grid(axis='y', alpha=0.3)

        plt.suptitle('Desempenho por Setor Economico', fontsize=12, fontweight='bold')
        plt.tight_layout()
        _salvar(fig, 'p41_b4_desempenho_setores.png')
        print()
        print('OK Figura salva: p41_b4_desempenho_setores.png')
else:
    print('AVISO: metricas_por_setor.csv nao disponivel.')


=== SMAPE medio por Setor x Variavel ===
SETOR               Commodities  Energia  Petróleo  Tecnologia  Varejo
Variavel                                                              
Ativo Circulante         0.1250   0.1270    0.1580      0.1100  0.1000
Ativo Total              0.1070   0.0970    0.1250      0.0850  0.1110
EBITDA                   0.1640   0.1400    0.1800      0.1800  0.1420
FCO                      0.8400   0.8720    1.1480      1.0030  1.5040
Lucro Líquido            1.1170   0.3430    0.9680      0.8310  0.8000
Passivo Circulante       0.1150   0.1460    0.1820      0.1390  0.1720
Passivo Total            0.1070   0.0970    0.1250      0.0850  0.1110
Patrimônio Líquido       0.1240   0.0850    0.1860      0.0870  0.0970
Receita Líquida          0.1390   0.1340    0.1640      0.1240  0.1820

Ranking de previsibilidade por setor (SMAPE medio):
  Energia            SMAPE=0.227  ███████████████████
  Tecnologia         SMAPE=0.294  █████████████████
  Commodities      

2026-06-07 23:21:05 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b4_desempenho_setores.png



OK Figura salva: p41_b4_desempenho_setores.png


##  Bloco 5 — Evolução do Z''-Score por Empresa

Exibe a trajetória do Z''-Score de Altman ao longo do tempo (2015–2025) para todas as
empresas. Empresas com Z'' caindo em direção às zonas cinza ou de insolvência merecem
atenção especial na análise de cenários.

In [7]:
if df_zscore is not None and not df_zscore.empty:
    df_z = df_zscore.copy()
    df_z.columns = df_z.columns.str.strip()

    # Normaliza coluna de zona
    if 'zona_altman' in df_z.columns:
        df_z['zona_altman'] = df_z['zona_altman'].str.strip()

    print('=== Z\'\'-Score — Distribuicao por Zona ===')
    if 'zona_altman' in df_z.columns:
        total = df_z['altman_z_pp'].notna().sum()
        for zona, cnt in df_z['zona_altman'].value_counts().items():
            pct = cnt / total * 100
            print(f'  {zona:<15} {cnt:>5} ({pct:5.1f}%)')

    if 'ANO' in df_z.columns:
        print()
        print('Z\'\'  medio por ano:')
        print(df_z.groupby('ANO')['altman_z_pp'].agg(['mean','median','std']).round(3).to_string())

    if 'SETOR' in df_z.columns:
        print()
        print('Z\'\'  medio por setor:')
        print(df_z.groupby('SETOR')['altman_z_pp'].agg(['mean','median','count']).round(3).to_string())

    # Nomes de empresas destaque
    emps_dest_nomes = set()
    for v in emps_destaque.values():
        nome = v['nome'] if isinstance(v, dict) else v
        emps_dest_nomes.add(nome)

    # Figura 4a — Evolucao Z'' por empresa (todas + destaque)
    if 'ANO' in df_z.columns and 'NOME_CIA' in df_z.columns:
        df_zv = df_z[df_z['altman_z_pp'].notna()].copy()
        emps_all = sorted(df_zv['NOME_CIA'].dropna().unique().tolist())

        fig, ax = plt.subplots(figsize=(14, 6))
        cs = plt.cm.tab20(np.linspace(0, 1, max(len(emps_all), 1)))

        for i, emp in enumerate(emps_all):
            s = df_zv[df_zv['NOME_CIA'] == emp].sort_values('ANO')
            if s.empty: continue
            destaque = emp in emps_dest_nomes
            ax.plot(
                s['ANO'], s['altman_z_pp'],
                marker='o' if destaque else None,
                label=emp[:16] if destaque else '_nolegend_',
                color=cs[i % len(cs)],
                linewidth=2.5 if destaque else 0.8,
                markersize=4 if destaque else 0,
                alpha=1.0 if destaque else 0.25,
                zorder=5 if destaque else 2
            )
            if destaque and len(s) > 0:
                ax.annotate(
                    emp[:14],
                    xy=(s['ANO'].iloc[-1], s['altman_z_pp'].iloc[-1]),
                    xytext=(4, 2), textcoords='offset points',
                    fontsize=6.5, color=cs[i % len(cs)], fontweight='bold'
                )

        ax.axhspan(df_zv['altman_z_pp'].min() - 0.5, ZONA_CINZA_INF,
                   alpha=0.07, color='#e74c3c', label='Zona Insolvencia')
        ax.axhspan(ZONA_CINZA_INF, ZONA_SEGURA,
                   alpha=0.07, color='#f39c12', label='Zona Cinza')
        ax.axhspan(ZONA_SEGURA, df_zv['altman_z_pp'].max() + 0.5,
                   alpha=0.04, color='#2ecc71', label='Zona Segura')
        ax.axhline(ZONA_CINZA_INF, ls='--', lw=1.2, color='#e67e22', alpha=0.7)
        ax.axhline(ZONA_SEGURA,    ls='--', lw=1.2, color='#27ae60', alpha=0.7)
        ax.set_xlabel('Ano', fontsize=9)
        ax.set_ylabel("Z''", fontsize=9)
        ax.set_title(
            "Evolucao do Z''-Score de Altman — Todas as Empresas (2015-2025)\n"
            "Linhas destacadas = empresas representativas por setor",
            fontsize=11, fontweight='bold'
        )
        ax.legend(fontsize=7, ncol=3, loc='upper left')
        ax.grid(alpha=0.2)
        plt.tight_layout()
        _salvar(fig, 'p41_b5_evolucao_zscore.png')
        print()
        print('OK Figura salva: p41_b5_evolucao_zscore.png')

    # Figura 4b — Ultimo Z'' por empresa (bar chart)
    if zscore_emp:
        df_ze = pd.DataFrame([
            {'empresa': v['nome'], 'setor': v['setor'],
             'z_ultimo': v['z_ultimo'], 'zona': v['zona_ultimo']}
            for v in zscore_emp.values()
            if isinstance(v, dict)
        ]).sort_values(['setor', 'z_ultimo'], ascending=[True, False])

        if not df_ze.empty:
            cores_zona = {
                'Segura': '#2ecc71', 'Cinza': '#f39c12',
                'Insolvencia': '#e74c3c', 'N/D': '#bdc3c7'
            }
            cores = df_ze['zona'].map(cores_zona).fillna('#bdc3c7')
            n = len(df_ze)
            fig, ax = plt.subplots(figsize=(max(10, 0.55 * n), 5))
            ax.barh(range(n), df_ze['z_ultimo'].values, color=cores.values, alpha=0.85)
            ax.axvline(ZONA_SEGURA,    ls='--', lw=1.2, color='#27ae60', label=f'Zona Segura ({ZONA_SEGURA})')
            ax.axvline(ZONA_CINZA_INF, ls='--', lw=1.2, color='#e67e22', label=f'Zona Cinza ({ZONA_CINZA_INF})')
            ax.set_yticks(range(n))
            ax.set_yticklabels([e[:20] for e in df_ze['empresa']], fontsize=7)
            ax.set_xlabel("Z'' (ultimo ano disponivel)", fontsize=9)
            ax.set_title(
                "Z''-Score Ultimo Ano por Empresa\nCores: verde=Segura | laranja=Cinza | vermelho=Insolvencia",
                fontsize=10, fontweight='bold'
            )
            ax.legend(fontsize=8)
            ax.grid(axis='x', alpha=0.25)
            plt.tight_layout()
            _salvar(fig, 'p41_b5_zscore_ultimo_empresa.png')
            print('OK Figura salva: p41_b5_zscore_ultimo_empresa.png')
else:
    print('AVISO: altman_zscore.parquet nao disponivel.')


=== Z''-Score — Distribuicao por Zona ===
  Segura            917 ( 88.9%)
  Cinza             112 ( 10.9%)
  Insolvencia         2 (  0.2%)

Z''  medio por ano:
       mean  median    std
ANO                       
2015 4.8810  4.6810 1.9480
2016 4.9930  4.3890 2.5160
2017 4.5350  4.2900 1.7480
2018 4.7970  4.4280 1.7880
2019 4.2260  3.9580 1.5570
2020 4.3870  4.4960 1.4780
2021 4.8200  4.7450 1.8570
2022 4.8990  4.8760 1.6100
2023 4.5720  4.3960 1.6150
2024 4.4380  4.2850 1.5330
2025 4.2840  4.2920 1.4340
2026 3.3000  3.4590 1.1340

Z''  medio por setor:
              mean  median  count
SETOR                            
Commodities 4.1080  4.0300    225
Energia     3.9130  3.7880    225
Petróleo    4.6930  4.1360    197
Tecnologia  5.2760  5.2770    177
Varejo      5.1450  5.1440    207


2026-06-07 23:21:05 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b5_evolucao_zscore.png



OK Figura salva: p41_b5_evolucao_zscore.png


2026-06-07 23:21:06 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b5_zscore_ultimo_empresa.png


OK Figura salva: p41_b5_zscore_ultimo_empresa.png


##  Bloco 6 — Distribuição de Incerteza Monte Carlo

Para cada empresa e variável, exibe os percentis P5, P50 e P95 das 500 simulações MC,
mostrando o range de cenários possíveis para 2026.

In [8]:
if df_stress is not None and not df_stress.empty:
    print('=== Resumo Monte Carlo — Coeficiente de Variacao por Variavel ===')
    print('CV = desvio / media: quanto maior, mais incerta a previsao')
    print()

    if 'variavel' in df_stress.columns and 'cv' in df_stress.columns:
        resumo_cv = df_stress.groupby('variavel')['cv'].agg(['mean','median','std']).round(3)
        print(resumo_cv.to_string())

    # Figura 5 — Intervalos MC P5-P95 por empresa (targets foco)
    for base_str, label in [('DRE_3.01', 'Receita Liquida'),
                              ('DRE_3.11', 'Lucro Liquido'),
                              ('EBITDA',   'EBITDA')]:
        t_alvo = f'TARGET_{base_str}_DFP'
        df_sf  = df_stress[df_stress['target'] == t_alvo].copy()
        if df_sf.empty:
            print(f'  AVISO: sem dados MC para {t_alvo}')
            continue

        df_sf = df_sf.sort_values(['setor', 'empresa']).reset_index(drop=True)
        n = len(df_sf)
        emps_dest_nomes_mc = {
            (v['nome'] if isinstance(v, dict) else v)
            for v in emps_destaque.values()
        }
        cores_bar = [
            '#e74c3c' if row['empresa'] in emps_dest_nomes_mc else '#3498db'
            for _, row in df_sf.iterrows()
        ]

        fig, ax = plt.subplots(figsize=(max(12, 0.6 * n), 5))
        x = np.arange(n)
        ax.bar(x, df_sf['p50'] / 1e6, color=cores_bar, alpha=0.75, label='Mediana MC (P50)')
        ax.errorbar(
            x, df_sf['p50'] / 1e6,
            yerr=[(df_sf['p50'] - df_sf['p5']) / 1e6,
                  (df_sf['p95'] - df_sf['p50']) / 1e6],
            fmt='none', color='#2c3e50', capsize=3, linewidth=1.2, label='IC [P5, P95]'
        )
        ax.scatter(x, df_sf['y_base'] / 1e6,
                   color='#f39c12', zorder=5, s=45, label='Base (sem perturbacao)')
        ax.set_xticks(x)
        ax.set_xticklabels(
            [e[:12] + '...' if len(e) > 12 else e for e in df_sf['empresa']],
            rotation=45, ha='right', fontsize=7
        )
        ax.set_ylabel('R$ bilhoes', fontsize=9)
        ax.set_title(
            f'Intervalos MC P5-P95 — {label} DFP 2026\n'
            'Barras vermelhas = empresas representativas por setor',
            fontsize=10, fontweight='bold'
        )
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        # Rotulos de setor
        if 'setor' in df_sf.columns:
            ylim_bot = ax.get_ylim()[0]
            for s in df_sf['setor'].dropna().unique():
                idx_s = df_sf[df_sf['setor'] == s].index.tolist()
                if idx_s:
                    mid = (idx_s[0] + idx_s[-1]) / 2
                    ax.text(mid, ylim_bot, s[:10], ha='center', fontsize=7,
                            color='#555', fontstyle='italic')

        plt.tight_layout()
        fname = f'p41_b6_mc_{base_str.replace(".", "_")}.png'
        _salvar(fig, fname)
        print(f'OK Figura salva: {fname}')

    # Figura 5d — CV por empresa (barras horizontais)
    if 'cv' in df_stress.columns and 'empresa' in df_stress.columns:
        df_cv_emp = (df_stress.groupby(['empresa', 'setor'])['cv']
                     .mean()
                     .reset_index()
                     .sort_values('cv', ascending=False)
                     .head(20))
        if not df_cv_emp.empty:
            fig, ax = plt.subplots(figsize=(10, 6))
            cores_s = [PALETA_SETOR.get(s, '#95a5a6') for s in df_cv_emp['setor']]
            ax.barh(range(len(df_cv_emp)), df_cv_emp['cv'].values,
                    color=cores_s, alpha=0.85)
            ax.set_yticks(range(len(df_cv_emp)))
            ax.set_yticklabels([e[:20] for e in df_cv_emp['empresa']], fontsize=8)
            ax.set_xlabel('CV medio (desvio / media)', fontsize=9)
            ax.set_title(
                'Top-20 Empresas com Maior Incerteza MC\n'
                '(Coeficiente de Variacao medio — targets DFP)',
                fontsize=10, fontweight='bold'
            )
            patches = [mpatches.Patch(color=v, label=k)
                       for k, v in PALETA_SETOR.items()
                       if k in df_cv_emp['setor'].values]
            if patches:
                ax.legend(handles=patches, fontsize=8, loc='lower right')
            ax.grid(axis='x', alpha=0.3)
            plt.tight_layout()
            _salvar(fig, 'p41_b6_cv_empresa.png')
            print('OK Figura salva: p41_b6_cv_empresa.png')
else:
    print('AVISO: analise_estresse_mc.csv nao disponivel.')


=== Resumo Monte Carlo — Coeficiente de Variacao por Variavel ===
CV = desvio / media: quanto maior, mais incerta a previsao

                     mean  median    std
variavel                                
Ativo Circulante   0.2010  0.1880 0.0660
Ativo Total        0.2030  0.2030 0.0050
EBITDA             0.1800  0.1820 0.0060
FCO                0.5200  0.3540 0.4420
Lucro Líquido      0.2970  0.2000 0.1560
Passivo Circulante 0.2070  0.2060 0.0070
Passivo Total      0.2030  0.2030 0.0050
Patrimônio Líquido 0.1570  0.1570 0.0070
Receita Líquida    0.2030  0.2030 0.0110


2026-06-07 23:21:06 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b6_mc_DRE_3_01.png


OK Figura salva: p41_b6_mc_DRE_3_01.png


2026-06-07 23:21:06 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b6_mc_DRE_3_11.png


OK Figura salva: p41_b6_mc_DRE_3_11.png


2026-06-07 23:21:07 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b6_mc_EBITDA.png


OK Figura salva: p41_b6_mc_EBITDA.png


2026-06-07 23:21:07 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b6_cv_empresa.png


OK Figura salva: p41_b6_cv_empresa.png


##  Bloco 7 — Cobertura Conformal por Variável

Valida os intervalos de predição com 90% de cobertura garantida. A cobertura real
deve estar próxima de 90% — desvios indicam intervalos mal calibrados.

In [9]:
if df_cp is not None and not df_cp.empty:
    print('=== Conformal Prediction — Cobertura Real vs. Esperada (90%) ===')
    print()

    _col_cob = 'CoberturaReal' if 'CoberturaReal' in df_cp.columns else None
    _col_lar = 'LarguraMedia'  if 'LarguraMedia'  in df_cp.columns else None
    _col_var = 'Variavel'      if 'Variavel'      in df_cp.columns else None

    if _col_cob and _col_var:
        resumo_cp = df_cp.groupby(_col_var)[_col_cob].agg(['mean','min','max']).round(3)
        print(resumo_cp.to_string())
        print()

        n_ok   = (df_cp[_col_cob] >= 0.85).sum()
        n_tot  = len(df_cp)
        n_bx   = (df_cp[_col_cob] < 0.85).sum()
        print(f'  Targets com cobertura >= 85%: {n_ok}/{n_tot}')
        print(f'  Targets com cobertura <  85%: {n_bx}/{n_tot} (intervalos sub-calibrados)')

    # Figura 6 — Scatter cobertura real vs. largura relativa
    if _col_cob and _col_lar and _col_var:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Cobertura por variavel
        ax = axes[0]
        df_cp_v = df_cp.groupby(_col_var).agg(
            cobertura=(_col_cob, 'mean'),
            largura=(_col_lar, 'mean')
        ).reset_index().sort_values('cobertura', ascending=False)
        cores_cob = ['#2ecc71' if v >= 0.88 else '#f39c12' if v >= 0.82 else '#e74c3c'
                     for v in df_cp_v['cobertura']]
        ax.barh(range(len(df_cp_v)), df_cp_v['cobertura'], color=cores_cob, alpha=0.85)
        ax.axvline(0.90, ls='--', lw=1.5, color='black', label='Alvo 90%')
        ax.axvline(0.85, ls=':',  lw=1.0, color='orange', label='Limite 85%')
        ax.set_yticks(range(len(df_cp_v)))
        ax.set_yticklabels(df_cp_v[_col_var], fontsize=8)
        ax.set_xlabel('Cobertura Real', fontsize=9)
        ax.set_title('Cobertura Conformal por Variavel\n(alvo = 90%)',
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1.05)
        ax.grid(axis='x', alpha=0.3)

        # Largura do intervalo em bilhoes
        ax2 = axes[1]
        df_cp_v_s = df_cp_v.sort_values('largura', ascending=False)
        ax2.barh(range(len(df_cp_v_s)), df_cp_v_s['largura'] / 1e6, color='#3498db', alpha=0.8)
        ax2.set_yticks(range(len(df_cp_v_s)))
        ax2.set_yticklabels(df_cp_v_s[_col_var], fontsize=8)
        ax2.set_xlabel('Largura Media do Intervalo (R$ bilhoes)', fontsize=9)
        ax2.set_title('Largura do Intervalo Conformal 90%\npor Variavel',
                      fontsize=10, fontweight='bold')
        ax2.grid(axis='x', alpha=0.3)

        plt.suptitle('Conformal Prediction — Calibracao e Largura dos Intervalos',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        _salvar(fig, 'p41_b7_conformal_cobertura.png')
        print()
        print('OK Figura salva: p41_b7_conformal_cobertura.png')
else:
    print('AVISO: conformal_summary.csv nao disponivel.')


=== Conformal Prediction — Cobertura Real vs. Esperada (90%) ===

                     mean    min    max
Variavel                               
Ativo Circulante   0.9160 0.9130 0.9170
Ativo Total        0.9160 0.9130 0.9170
EBITDA             0.9030 0.8840 0.9170
FCO                0.9150 0.8740 0.9580
Lucro Líquido      0.9020 0.8840 0.9170
Passivo Circulante 0.9160 0.9130 0.9170
Passivo Total      0.9160 0.9130 0.9170
Patrimônio Líquido 0.9160 0.9130 0.9170
Receita Líquida    0.9290 0.8960 0.9580

  Targets com cobertura >= 85%: 36/36
  Targets com cobertura <  85%: 0/36 (intervalos sub-calibrados)


2026-06-07 23:21:07 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b7_conformal_cobertura.png



OK Figura salva: p41_b7_conformal_cobertura.png


##  Bloco 8 — Feature Importance: Quais Variáveis Explicam as Previsões?

Mostra quais features os modelos mais utilizam para fazer previsões.
Features de lag (valores defasados do próprio target) normalmente dominam, o que
é esperado em séries financeiras com alta autocorrelação.

In [10]:
if df_feat is not None and not df_feat.empty:
    print('=== Top-25 Features por Importancia Agregada (todos os 36 targets) ===')

    if 'feature' in df_feat.columns and 'importancia_total' in df_feat.columns:
        top25 = df_feat.head(25)
        print(top25[['rank','feature','familia','importancia_total']].to_string(index=False))

        print()
        print('Importancia por familia:')
        if 'familia' in df_feat.columns:
            print(df_feat.groupby('familia')['importancia_total']
                  .agg(['sum','count'])
                  .sort_values('sum', ascending=False)
                  .round(4).to_string())

    # Figura 7a — Barras horizontais Top-25
    CORES_FAM = {
        'Lag/Roll': '#3498db', 'KPI base': '#f39c12', 'YoY': '#2ecc71',
        'Macro': '#e74c3c', 'Razao/Cruzada': '#1abc9c', 'Setor dummy': '#9b59b6',
        'Posicao Setor': '#e67e22', 'Sazonalidade': '#95a5a6',
        'Temporal': '#bdc3c7', 'Outro': '#ecf0f1'
    }

    top = df_feat.head(25).copy()
    cores = top['familia'].map(CORES_FAM).fillna('#bdc3c7') if 'familia' in top.columns else ['#3498db'] * len(top)
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(range(len(top)), top['importancia_total'].values[::-1],
            color=list(cores)[::-1], alpha=0.85)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['feature'].values[::-1], fontsize=8)
    ax.set_xlabel('Importancia Agregada (36 targets)', fontsize=10)
    ax.set_title('Top-25 Features por Importancia Agregada\n'
                 '(Media de importancias de todos os 36 targets treinados)',
                 fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    if 'familia' in top.columns:
        patches = [mpatches.Patch(color=v, label=k)
                   for k, v in CORES_FAM.items()
                   if k in top['familia'].values]
        if patches:
            ax.legend(handles=patches, fontsize=8, loc='lower right')
    plt.tight_layout()
    _salvar(fig, 'p41_b8_feature_importance.png')
    print()
    print('OK Figura salva: p41_b8_feature_importance.png')

    # Figura 7b — Pizza por familia
    if 'familia' in df_feat.columns:
        fam_agg = (df_feat.groupby('familia')['importancia_total']
                   .sum()
                   .sort_values(ascending=False))
        cores_pizza = [CORES_FAM.get(f, '#bdc3c7') for f in fam_agg.index]
        fig, ax = plt.subplots(figsize=(7, 7))
        wedges, texts, autotexts = ax.pie(
            fam_agg.values, labels=fam_agg.index,
            colors=cores_pizza, autopct='%1.1f%%',
            startangle=90, pctdistance=0.8
        )
        for t in texts + autotexts:
            t.set_fontsize(9)
        ax.set_title('Importancia por Familia de Features\n(todos os 36 targets)',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        _salvar(fig, 'p41_b8_feature_familia_pizza.png')
        print('OK Figura salva: p41_b8_feature_familia_pizza.png')
else:
    print('AVISO: feature_importance_ranking.csv nao disponivel.')


=== Top-25 Features por Importancia Agregada (todos os 36 targets) ===
 rank                           feature  familia  importancia_total
    1          TARGET_BPA_1_ITR_T3_lag1 Lag/Roll             3.7225
    2       TARGET_BPA_1.01_ITR_T3_lag1 Lag/Roll             1.9242
    3       TARGET_BPP_2.03_ITR_T3_lag1 Lag/Roll             1.8485
    4          TARGET_BPP_2_ITR_T2_lag2 Lag/Roll             1.7600
    5          TARGET_BPP_2_ITR_T3_lag2 Lag/Roll             1.6750
    6       TARGET_DRE_3.01_ITR_T3_lag1 Lag/Roll             1.3512
    7       TARGET_BPA_1.01_ITR_T1_lag1 Lag/Roll             1.2762
    8       TARGET_BPP_2.01_ITR_T2_lag1 Lag/Roll             1.0455
    9       TARGET_BPA_1.01_ITR_T2_lag1 Lag/Roll             1.0230
   10       TARGET_BPP_2.01_ITR_T3_lag1 Lag/Roll             0.9652
   11       TARGET_BPP_2.03_ITR_T1_lag1 Lag/Roll             0.9553
   12       TARGET_BPP_2.03_ITR_T2_lag1 Lag/Roll             0.9176
   13       TARGET_BPP_2.01_ITR_T2_lag2 Lag/R

2026-06-07 23:21:08 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b8_feature_importance.png
2026-06-07 23:21:08 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b8_feature_familia_pizza.png



OK Figura salva: p41_b8_feature_importance.png
OK Figura salva: p41_b8_feature_familia_pizza.png


##  Bloco 9 — Análise de Resíduos: Viés e Distribuição dos Erros

Avalia se os modelos têm viés sistemático (tendência a sub ou superestimar)
e como os erros se distribuem por variável e horizonte.

In [11]:
if df_res is not None and not df_res.empty:
    print('=== Resumo de Residuos por Variavel x Horizonte ===')

    if 'Variavel' in df_res.columns and 'SMAPE' in df_res.columns:
        # Pivô SMAPE por variavel x horizonte
        if 'Horizonte' in df_res.columns:
            df_res['Horizonte_clean'] = df_res['Horizonte'].str.replace('_', '')
            pv_r = df_res.pivot_table(
                values='SMAPE', index='Variavel',
                columns='Horizonte_clean', aggfunc='mean'
            )
            col_order = [c for c in ['ITRT1','ITRT2','ITRT3','DFP'] if c in pv_r.columns]
            print(pv_r[col_order].round(3).to_string())
            print()

        # Vies: superestimacao
        if 'superestimacao_pct' in df_res.columns:
            print('Taxa de superestimacao (modelo previu MAIS que o real):')
            print(df_res.groupby('Variavel')['superestimacao_pct']
                  .mean().sort_values(ascending=False).round(3).to_string())

    # Figura 8 — Heatmap de SMAPE dos residuos
    if 'Variavel' in df_res.columns and 'SMAPE' in df_res.columns and 'Horizonte' in df_res.columns:
        df_res_plot = df_res.copy()
        df_res_plot['Horizonte_clean'] = df_res_plot['Horizonte'].str.replace('_', '')
        pv_fig = df_res_plot.pivot_table(
            values='SMAPE', index='Variavel',
            columns='Horizonte_clean', aggfunc='mean'
        )
        col_order = [c for c in ['ITRT1','ITRT2','ITRT3','DFP'] if c in pv_fig.columns]
        pv_fig = pv_fig[col_order]

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Heatmap SMAPE
        vmax_r = min(float(pv_fig.max().max()), 1.0)
        sns.heatmap(pv_fig, annot=True, fmt='.3f', cmap='RdYlGn_r',
                    linewidths=0.5, linecolor='white', ax=axes[0],
                    vmin=0, vmax=vmax_r,
                    cbar_kws={'label': 'SMAPE medio'})
        axes[0].set_title('SMAPE dos Residuos\npor Variavel x Horizonte',
                           fontsize=10, fontweight='bold')
        axes[0].set_xlabel('Horizonte')

        # Heatmap taxa de superestimacao
        if 'superestimacao_pct' in df_res_plot.columns:
            pv_sup = df_res_plot.pivot_table(
                values='superestimacao_pct', index='Variavel',
                columns='Horizonte_clean', aggfunc='mean'
            )
            pv_sup = pv_sup[[c for c in col_order if c in pv_sup.columns]]
            sns.heatmap(pv_sup, annot=True, fmt='.2f', cmap='RdYlBu_r',
                        linewidths=0.5, linecolor='white', ax=axes[1],
                        vmin=0, vmax=1,
                        cbar_kws={'label': 'Proporcao superestimado'})
            axes[1].set_title('Taxa de Superestimacao\n(> 0.5 = modelo tende a prever alto)',
                               fontsize=10, fontweight='bold')
            axes[1].set_xlabel('Horizonte')

        plt.suptitle('Analise de Residuos — Qualidade das Previsoes', fontsize=12, fontweight='bold')
        plt.tight_layout()
        _salvar(fig, 'p41_b9_residuos_heatmap.png')
        print()
        print('OK Figura salva: p41_b9_residuos_heatmap.png')
else:
    print('AVISO: residuos_resumo.csv nao disponivel.')


=== Resumo de Residuos por Variavel x Horizonte ===
Horizonte_clean     ITRT1  ITRT2  ITRT3    DFP
Variavel                                      
Ativo Circulante   0.1210 0.1120 0.0870 0.1760
Ativo Total        0.1000 0.0750 0.0650 0.1830
EBITDA             0.2710 0.1600 0.0980 0.1110
FCO                1.2270 1.0770 1.2260 0.7740
Lucro Líquido      0.7090 0.9010 1.0130 0.6210
Passivo Circulante 0.1240 0.1320 0.1220 0.2290
Passivo Total      0.1000 0.0750 0.0650 0.1830
Patrimônio Líquido 0.1030 0.0810 0.0940 0.1890
Receita Líquida    0.1350 0.1300 0.1260 0.2060

Taxa de superestimacao (modelo previu MAIS que o real):
Variavel
Ativo Circulante     0.4870
Patrimônio Líquido   0.4660
Passivo Circulante   0.4360
EBITDA               0.4160
Lucro Líquido        0.4110
Receita Líquida      0.3910
Ativo Total          0.3810
Passivo Total        0.3810
FCO                  0.3490


2026-06-07 23:21:08 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b9_residuos_heatmap.png



OK Figura salva: p41_b9_residuos_heatmap.png


##  Bloco 10 — Painel Predito × Observado (Targets Foco TCC)

Compara diretamente os valores previstos pelo modelo com os valores reais
para os 12 targets foco do TCC (Receita, Lucro, EBITDA × 4 horizontes).

In [12]:
import numpy as np

_TARGET_BASES = ['DRE_3.01', 'DRE_3.11', 'EBITDA',
                 'BPA_1', 'BPA_1.01', 'BPP_2.01', 'BPP_2.03', 'BPP_2', 'DFC_MI_6.01']
_HORIZONTES   = ['_ITR_T1', '_ITR_T2', '_ITR_T3', '_DFP']
_LOG_BASES    = {'DRE_3.01','EBITDA','BPA_1','BPA_1.01','BPP_2.01','BPP_2.03','BPP_2'}
_ARC_BASES    = {'DFC_MI_6.01','DRE_3.11'}
LOG_TARGETS   = {f'TARGET_{b}{h}' for b in _LOG_BASES for h in _HORIZONTES}
ARC_TARGETS   = {f'TARGET_{b}{h}' for b in _ARC_BASES  for h in _HORIZONTES}

def get_transform(target):
    if target in LOG_TARGETS:    return 'log1p'
    if target in ARC_TARGETS:    return 'arcsinh'
    return 'none'

def inv_transform(y, t='none'):
    y = np.asarray(y, float)
    if t == 'log1p':   return np.expm1(y)
    if t == 'arcsinh': return np.sinh(y)
    return y

BASES_FOCO   = ['DRE_3.01', 'DRE_3.11', 'EBITDA']
TARGETS_FOCO = [f'TARGET_{b}{h}' for b in BASES_FOCO for h in _HORIZONTES]

def _smape(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    d = (np.abs(yt) + np.abs(yp)) / 2.0
    m = d > 1e-9
    return float(np.mean(np.abs(yt[m] - yp[m]) / d[m])) if m.sum() > 0 else np.nan

def _r2(yt, yp):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    if len(yt) < 2 or np.isclose(np.var(yt), 0): return np.nan
    ss_r = np.sum((yt - yp) ** 2)
    ss_t = np.sum((yt - yt.mean()) ** 2)
    return float(1 - ss_r / ss_t) if ss_t > 0 else np.nan

if df_pred is not None and not df_pred.empty:
    df_fp = df_pred[
        df_pred['Target'].isin(TARGETS_FOCO) &
        df_pred.apply(lambda r: melhores.get(r['Target']) == r['Algoritmo'], axis=1)
    ].copy()

    tgts_plot = [t for t in TARGETS_FOCO if t in df_fp['Target'].values]

    if tgts_plot:
        ncols = 4
        nrows = int(np.ceil(len(tgts_plot) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
        axes = np.array(axes).reshape(-1)

        for idx, target in enumerate(tgts_plot):
            ax  = axes[idx]
            sub = df_fp[df_fp['Target'] == target].dropna(subset=['y_true', 'y_pred'])
            if sub.empty:
                ax.set_visible(False)
                continue

            # y_true em escala transformada, y_pred em R$ original
            transf = get_transform(target)
            yt = inv_transform(sub['y_true'].values, transf)
            yp = sub['y_pred'].values.astype(float)

            mask = np.isfinite(yt) & np.isfinite(yp)
            if mask.sum() < 2:
                ax.set_visible(False)
                continue

            yt, yp = yt[mask], yp[mask]
            alg  = melhores.get(target, '?')
            r2v  = _r2(yt, yp)
            spv  = _smape(yt, yp)

            ax.scatter(yp, yt, alpha=0.5, s=16, edgecolors='none',
                       color=PALETA_ALG.get(alg, '#3498db'))
            lm = min(yt.min(), yp.min()); lM = max(yt.max(), yp.max())
            ax.plot([lm, lM], [lm, lM], 'k--', lw=0.9, alpha=0.6)

            base = target.replace('TARGET_','').rsplit('_ITR',1)[0].rsplit('_DFP',1)[0]
            hor  = next((h.replace('_','') for h in _HORIZONTES if target.endswith(h)), '')
            ax.set_title(
                f'{NOME_VARIAVEL.get(base, base)} [{hor}]\n{alg}  R²={r2v:.3f}  SMAPE={spv:.1%}',
                fontsize=8
            )
            ax.set_xlabel('Predito (R$ bilhoes)', fontsize=7)
            ax.set_ylabel('Observado (R$ bilhoes)', fontsize=7)
            ax.tick_params(labelsize=7)
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}'))
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}'))

        for j in range(len(tgts_plot), len(axes)):
            axes[j].set_visible(False)

        plt.suptitle(
            'Predito x Observado — Targets Foco TCC (Receita, Lucro, EBITDA)\n'
            'Hold-out 2024-2025 | Melhor modelo por target',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        _salvar(fig, 'p41_b10_predito_vs_observado.png')
        print('OK Figura salva: p41_b10_predito_vs_observado.png')
    else:
        print('AVISO: nenhum target foco com predicoes disponivel em df_pred.')
else:
    print('AVISO: predicoes_teste_detalhadas.parquet nao disponivel.')


2026-06-07 23:21:09 | INFO     | Figura salva: outputs\painel_avaliacao\p41_b10_predito_vs_observado.png


OK Figura salva: p41_b10_predito_vs_observado.png


##  Bloco 11 — Resumo Final e Lista de Arquivos Gerados

In [13]:
from datetime import datetime

print('=' * 70)
print('RESUMO FINAL — 04_1_cvm_avaliacao_painel.ipynb')
print('=' * 70)
print(f'Executado em: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()

# Lista todas as figuras geradas
figuras_geradas = sorted(PASTA_PAINEL.glob('p41_*.png'))
print(f'Figuras geradas: {len(figuras_geradas)}')
for f in figuras_geradas:
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} ({kb:.0f} KB)')

print()
print('Figuras salvas em:', PASTA_PAINEL)
print()
print('Para usar as figuras no TCC:')
print('  - p41_b2_smape_heatmap.png          => Tabela de desempenho geral')
print('  - p41_b3_overfitting_scatter.png    => Diagnostico de overfitting')
print('  - p41_b4_desempenho_setores.png     => Analise setorial')
print('  - p41_b5_evolucao_zscore.png        => Evolucao do risco (Z\'\')')
print('  - p41_b6_mc_*.png                   => Incerteza Monte Carlo')
print('  - p41_b7_conformal_cobertura.png    => Calibracao dos intervalos')
print('  - p41_b8_feature_importance.png     => Variaveis mais relevantes')
print('  - p41_b9_residuos_heatmap.png       => Qualidade das previsoes')
print('  - p41_b10_predito_vs_observado.png  => Predito x Real (foco TCC)')
print()
print('OK Script 4.1 concluido')


RESUMO FINAL — 04_1_cvm_avaliacao_painel.ipynb
Executado em: 2026-06-07 23:21:09

Figuras geradas: 14
  p41_b10_predito_vs_observado.png              (16 KB)
  p41_b2_smape_heatmap.png                      (98 KB)
  p41_b3_overfitting_scatter.png                (84 KB)
  p41_b4_desempenho_setores.png                 (84 KB)
  p41_b5_evolucao_zscore.png                    (369 KB)
  p41_b5_zscore_ultimo_empresa.png              (71 KB)
  p41_b6_cv_empresa.png                         (66 KB)
  p41_b6_mc_DRE_3_01.png                        (92 KB)
  p41_b6_mc_DRE_3_11.png                        (96 KB)
  p41_b6_mc_EBITDA.png                          (90 KB)
  p41_b7_conformal_cobertura.png                (84 KB)
  p41_b8_feature_familia_pizza.png              (68 KB)
  p41_b8_feature_importance.png                 (121 KB)
  p41_b9_residuos_heatmap.png                   (167 KB)

Figuras salvas em: outputs\painel_avaliacao

Para usar as figuras no TCC:
  - p41_b2_smape_heatmap.png        